# Running ECMWF AIFS on Any Machine

**ECMWF's AIFS** (Artificial Intelligence Forecast System) is a state-of-the-art
ML weather model — but its official implementation requires **Ampere-class NVIDIA GPUs**
and the `flash-attn` library, which is painful to install.

This notebook shows how to run AIFS on:

| Hardware | What happens |
|---|---|
| **NVIDIA Ampere / Ada (A100, H100, RTX 3090+)** | Full speed, best performance |
| **Older NVIDIA (V100, T4, RTX 20xx)** | Works via PyTorch SDPA fallback |
| **Apple Silicon (M1/M2/M3)** | Runs via Metal (MPS) — chunked attention |
| **CPU only** | Works but slow; fine for testing |

---
**What you'll need**
- Python ≥ 3.10
- The packages in `requirements.txt`
- ~1–2 GB disk space for the IC cache
- Internet access (to download ECMWF Open Data)


## 1. Installation

```bash
git clone https://huggingface.co/datasets/YOUR_USERNAME/aifs-tutorial
cd aifs-tutorial
pip install -r requirements.txt
```

> **Note on `flash-attn`**: you do **not** need to install it.
> This repo ships a pure-PyTorch shim that replaces it automatically.


## 2. Check your device


In [ ]:
from aifs.device import get_device, device_label

print("Active device :", device_label())
print("Device string :", get_device())  # 'cuda' | 'mps' | 'cpu'


## 3. Download initial conditions

AIFS uses **two consecutive 6-hour analysis fields** (t-6h and t) as input.
We pull them from [ECMWF Open Data](https://www.ecmwf.int/en/forecasts/datasets/open-data),
which is freely available without registration.

Multiple datasets are downloaded at this step. Once downloaded for the first time, each dataset is stored in cache.
The first download takes a few minutes.
Then, loading the data from the cache only takes a few seconds.

During the initial download, ratelimits and errors sometimes occur. If you're getting an error during the downloading of the data, just wait a few seconds and try again.


In [ ]:
from aifs.initial_conditions import load_ics

fields, date = load_ics(variant="single", cache_dir="ic_cache")

print(f"\nInitialisation date : {date}")
print(f"Number of fields    : {len(fields)}")
print(f"Example field shape : {list(fields.values())[0].shape}")


## 4.  Run a forecast

`lead_time` must be a multiple of 6 hours.
Each step takes ~20–60 s on a modern GPU, ~5–15 min on CPU.

**Memory guidance for `num_chunks`**:
The highest `num_chunks` is, the least memory is used. The inference will be slightly slower (16 is a good start, change to 32 or 64 if running into RAM issues).


In [ ]:
from aifs.forecast import run_forecast, load_forecast

states = run_forecast(
    fields=fields,
    date=date,
    lead_time=24,
    num_chunks=16,
    checkpoint="aifs-single-2.0"
)

print(f"\nOutput steps : {len(states)}")
for s in states:
    print(f"  {s['date']}")

## 5.  Inspect raw output

Each `state` is a plain dict.  The fields live under `state["fields"]`.


In [ ]:
step = states[0]
t2m  = step["fields"]["2t"]              # 2-m temperature

print("Step date      :", step["date"])
print("Field keys     :", list(step["fields"].keys())[:10], "…")
print("2t shape       :", t2m.shape)
print(f"2t range       : {t2m.min():.1f} – {t2m.max():.1f} K")
print(f"2t global mean : {t2m.mean():.2f} K  ({t2m.mean() - 273.15:.2f} °C)")


## 6. Plot a single field

In [ ]:
from aifs.plot import plot_field

fig = plot_field(states[-1], "2t")
fig.savefig("t2m.png", dpi=150, bbox_inches="tight")

### Try other variables:

In [ ]:
fig = plot_field(states[-1], "10u")
fig.savefig("10u.png", dpi=150, bbox_inches="tight")

In [ ]:
for i in range(4):
    fig = plot_field(states[i], "100u")

## 7 — Streaming forecast (for long runs / GUIs)

Use `run_forecast_streaming` to process each step as it arrives instead of waiting for the full run to complete.


In [ ]:
from aifs.forecast import run_forecast_streaming

for state in run_forecast_streaming(fields, date, lead_time=24):
    t2m_mean = state["fields"]["2t"].mean() - 273.15
    print(f"  {state['date']}  |  global mean T2m = {t2m_mean:.2f} °C")


## 9. Saving & loading forecast output (saved locally)


In [ ]:
import os
from pathlib import Path

FORECASTS = Path(".") / "forecasts"

FORECASTS.mkdir(exist_ok=True)

In [ ]:
from aifs.forecast import save_forecast

date = states[0]['date']
path = FORECASTS / f"output_{date}.nc"
save_forecast(states, path=path)

In [ ]:
from aifs.forecast import load_forecast
from aifs.plot import plot_field

states = load_forecast(path=path)

fig = plot_field(states[-1], "2t")
fig.savefig("t2m.png", dpi=150, bbox_inches="tight")

## 10. Loading forecast output from Hugging Face


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from aifs.forecast import load_forecast
from aifs.plot import plot_field

api = HfApi()
files = api.list_repo_files(
    repo_id="EmmaScharfmann/aifs-results",
    repo_type="dataset",
)
print(files)

f = FORECASTS / "20260703T0600/state_20260705T0600.npz"

local_path = hf_hub_download(
        repo_id="EmmaScharfmann/aifs-results",
        repo_type="dataset",
        filename=f,
    )

states = load_forecast(path=local_path)
for i in range(len(states)):
    fig = plot_field(states[i], "2t")


## Next steps

- **Longer forecasts**: change `lead_time` to 72, 120, 240 hours
- **Regional zoom**: use Cartopy's `set_extent` on the returned figure's axes
- **NetCDF export**: regrid from N320 back to a lat/lon grid with `earthkit-regrid`
  then write with `xarray` + `netCDF4`
- **Gradio demo**: see `app.py` in this repo for a browser-based interface
- **AIFS Ensemble**: swap the checkpoint for `ecmwf/aifs-ens-1.0`

Questions / issues?  Open an issue on the HuggingFace repo.
